# Smoke tests for `geolab-gpu2`

For every package in `environment.yml` (conda + pip): try to import it and
exercise one minimal API call. CLI-only packages get a `which`/`--version`
check instead. A failure here means something installed but doesn't load,
which is usually a sign of an ABI mismatch or a missing system library.

Run all cells. The summary at the bottom lists pass/fail per package.

## Setup

In [ ]:
import sys

import test_helpers as test
from test_helpers import RESULTS, cli, py, functional

test.RESULTS.clear()

print(f'Python {sys.version}')
print(f'sys.prefix: {sys.prefix}')

## Cloud & storage

In [ ]:
cli('aws')
py('awswrangler')
py('boto3', smoke=lambda m: m.client('s3', region_name='us-east-1'))
py('fsspec', smoke=lambda m: m.filesystem('memory'))
py('obstore')
py('s3fs', smoke=lambda m: m.S3FileSystem)

## Geospatial

In [ ]:
py('cartopy.crs', alias='cartopy',
   smoke=lambda m: m.PlateCarree())
py('contextily')
py('fiona', smoke=lambda m: m.supported_drivers)
py('folium',
   smoke=lambda m: m.Map(location=[0, 0], zoom_start=2))
py('osgeo.gdal', alias='gdal',
   smoke=lambda m: m.VersionInfo('RELEASE_NAME'))
py('ipyleaflet', smoke=lambda m: m.Map())
py('lonboard')
py('pyproj', smoke=lambda m: m.CRS('EPSG:4326'))
py('shapely.geometry', alias='shapely',
   smoke=lambda m: m.Point(0, 0).buffer(1).area)

## Core scientific stack

In [ ]:
py('numpy', smoke=lambda m: int(m.array([1, 2, 3]).sum()))
py('numba', smoke=lambda m: m.njit(lambda x: x + 1)(1))
py('scipy.stats', alias='scipy', smoke=lambda m: m.norm.cdf(0))
py('pandas',
   smoke=lambda m: m.DataFrame({'a': [1, 2]}).shape)
py('geopandas')
import matplotlib; matplotlib.use('Agg')
py('matplotlib', alias='matplotlib-base',
   smoke=lambda m: m.figure.Figure())
cli('dot', version_flag='-V')
py('pygraphviz', smoke=lambda m: m.AGraph().add_node(1))
py('xarray',
   smoke=lambda m: m.DataArray([1, 2, 3]).sum().item())
py('netCDF4', alias='netcdf4')
py('h5py')
py('h5netcdf')
py('pyarrow',
   smoke=lambda m: m.array([1, 2, 3]).to_pylist())
py('zarr',
   smoke=lambda m: m.zeros((3,), chunks=3, dtype='f4'))
py('virtualizarr')
py('bottleneck',
   smoke=lambda m: m.nansum([1.0, 2.0, float('nan'), 3.0]))
py('flox')
py('pooch')
py('dask.array', alias='dask',
   smoke=lambda m: m.ones(10, chunks=5).sum().compute())
py('distributed')
py('dask_gateway', alias='dask-gateway')
py('cvxpy', smoke=lambda m: m.Variable(name='x'))

## Geo / geoscience

In [ ]:
py('dascore')
cli('gmt', version_flag='--version')
py('obspy',
   smoke=lambda m: m.UTCDateTime('2020-01-01').timestamp)
py('obsplus')
py('pygmt')

## GPU / machine learning

In [ ]:
# PyTorch & seismic ML -----------------------------------------
# Informational only, not a pass/fail check: the driver is baked
# into this image, but the GPU device nodes still have to be
# passed through at `docker run` (--gpus all) or by the scheduler,
# so CUDA can legitimately be unavailable even on a correct build.
import warnings

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')

py('torch',
   smoke=lambda m: m.tensor([1., 2., 3.]).sum().item())
py('seisbench.models', alias='seisbench',
   smoke=lambda m: m.PhaseNet())

# Geometric / graph deep learning ---------------------------------
# torchvision/torchaudio: pinned to match `torch` exactly (see
# requirements.txt), smoke-tested with one op each that doesn't need
# any real image/audio data.
py('torchvision',
   smoke=lambda m: m.transforms.ToTensor()(m.transforms.functional.to_pil_image(torch.zeros(3, 4, 4))))
py('torchaudio',
   smoke=lambda m: m.transforms.Resample(16000, 8000)(torch.zeros(1, 16000)))
py('networkx', smoke=lambda m: m.Graph([(1, 2), (2, 3)]))
# torch_geometric.nn.pool.select.base calls torch.jit.script(...) at
# *import* time (a long-standing pattern for a TorchScript-compatible
# output class), and this pinned torch has started emitting a
# FutureWarning for any use of torch.jit.script at all, not just ours.
# Nothing in this repo calls torch.jit.script, and there's no way to
# avoid it short of forking torch_geometric, so it's suppressed
# narrowly (this exact warning, from that exact upstream module) rather
# than silencing warnings generally -- a real warning from torch_geometric
# itself would still surface.
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=FutureWarning, module='torch.jit._script')
    py('torch_geometric',
       smoke=lambda m: m.data.Data(x=torch.rand(3, 2), edge_index=torch.tensor([[0, 1], [1, 2]]).t()))
# torch_scatter/torch_sparse/torch_cluster: built from source in the
# Dockerfile (no prebuilt wheel for the pinned torch, see the RUN
# block there) -- these compile CPU-only extensions in this image, so
# the smoke calls below run on CPU regardless of GPU availability.
py('torch_scatter',
   smoke=lambda m: m.scatter_add(torch.tensor([1., 2., 3., 4.]), torch.tensor([0, 0, 1, 1])))
py('torch_sparse',
   smoke=lambda m: m.SparseTensor(row=torch.tensor([0, 1]), col=torch.tensor([1, 0])))
py('torch_cluster',
   smoke=lambda m: m.radius_graph(torch.rand(4, 2), r=1.5))

# General ML / GPU tooling --------------------------------------
py('sklearn', alias='scikit-learn',
   smoke=lambda m: m.linear_model.LinearRegression())
py('skimage', alias='scikit-image',
   smoke=lambda m: m.data.camera())
py('xgboost',
   smoke=lambda m: m.XGBRegressor())
py('dask_ml', alias='dask-ml')
py('seaborn')
py('jupyterlab_nvdashboard', alias='jupyterlab-nvdashboard',
   smoke=lambda m: m._jupyter_labextension_paths())

# Profiling ------------------------------------------------------
py('line_profiler')
py('memory_profiler')
py('snakeviz')
py('watermark')

### GPU compute test

The checks above only prove the packages *import*, they pass on CPU-only
hosts too. This cell actually runs a computation on the GPU and fails loudly
if there isn't one to run it on, unlike the informational check above, this
one is a real pass/fail test.

Run this after `docker run --gpus all ...` (needs `nvidia-container-toolkit`
on the host), or in a GeoLab session that was actually scheduled onto a GPU
node.

In [ ]:
assert torch.cuda.is_available(), (
    'No GPU visible to this container. Re-run with `docker run --gpus all ...` '
    '(needs nvidia-container-toolkit on the host), or confirm a GPU was '
    'actually scheduled onto this session.'
)

device = torch.device('cuda')
a = torch.randn(4096, 4096, device=device)
b = torch.randn(4096, 4096, device=device)
c = a @ b
torch.cuda.synchronize()  # the matmul above is async; wait for it to finish

assert c.is_cuda
print(f'Ran a 4096x4096 matmul on: {torch.cuda.get_device_name(0)}')
print(f'Result: device={c.device}, shape={tuple(c.shape)}, c[0, 0]={c[0, 0].item():.4f}')
print('GPU compute test passed.')

## Phase association & location

Tools for grouping picks (from seisbench or any other picker) into events
and locating them, none of these run the actual picking above. `pyocto`,
`gamma` (GaMMA), and `pyrocko` are pip packages, checked with a plain
`py()` import/smoke call. `pick2real`, `velest`, `Vel2Grid`, `Grid2Time`,
and `ph2dt` (helper/intermediate binaries in the REAL and NLLoc/HypoDD
pipelines) get a `cli()` check, on `$PATH` and responds at all.

`REAL`, `NLLoc` (NonLinLoc), `hypoDD`, and `hyp1.40` (HypoInverse), the
four tools that actually do association/location, get more than that.
`REAL`, `NLLoc`, and `hypoDD` each write a small synthetic dataset
(station geometry, a known true hypocenter, and P-wave arrival times
computed by hand from that geometry and a fixed velocity), run the real
binary on it in a temp directory, and assert the tool recovers the
known-true location/relocation to within a stated tolerance. `hyp1.40`
instead runs HypoInverse's own bundled `testone` reference case (a real
Geysers-area earthquake, baked into the image, see the Dockerfile) and
checks the output matches its known-correct summary line byte-for-byte
-- no need to hand-derive a "true" answer when the tool ships one.
A binary that's merely on `$PATH` but broken (bad build, wrong file
format expectations, a silently-dropped pick) fails these; it wouldn't
have failed a `cli()` check. See the comment above each test for the
exact scenario and how the expected values were derived.

In [ ]:
# Association ----------------------------------------------------
py('pyocto',
   smoke=lambda m: m.VelocityModel0D(p_velocity=6.0, s_velocity=3.5, tolerance=2.0))
py('gamma', alias='gmma',
   smoke=lambda m: m.GaussianMixture(n_components=1,
                                      station_locs=[[0.0, 0.0, 0.0], [1.0, 1.0, 0.0]],
                                      phase_type=['p', 's']))

# Waveform / location toolkit --------------------------------------
py('pyrocko')

# Helper/intermediate binaries (built from source, see Dockerfile) --
cli('pick2real')
cli('velest')
cli('Vel2Grid')
cli('Grid2Time')
cli('ph2dt')


import math
import subprocess
import tempfile
from pathlib import Path

KM_PER_DEG = 111.19  # good enough at these latitudes for a synthetic test


def _travel_time(elat, elon, edepth, slat, slon, vp):
    """Straight-line travel time in a homogeneous velocity model."""
    dlat = (elat - slat) * KM_PER_DEG
    dlon = (elon - slon) * KM_PER_DEG * math.cos(math.radians((elat + slat) / 2))
    return math.hypot(math.hypot(dlat, dlon), edepth) / vp


# REAL: association + location -------------------------------------
# One synthetic event at (lat 42.80, lon 13.20, depth 8.0 km), origin
# 2016-10-14 00:01:40.000 UTC (100.000 s into the day), Vp/Vs 6.2/3.3.
# 5 stations at asymmetric distances (symmetric placement gives two
# stations numerically-identical arrival times, which REAL's duplicate
# -pick filter silently drops, an easy trap when hand-building this).
# REAL also needs both P and S picks per station to associate at all,
# a P-only pick set gets silently discarded before the first selection
# stage. Each arrival = origin + travel_time(event, station, phase velocity).
def _test_real():
    stations = {
        'STA1': (42.8500, 13.1500), 'STA2': (42.7600, 13.2600),
        'STA3': (42.8200, 13.3100), 'STA4': (42.7400, 13.1300),
        'STA5': (42.9000, 13.2300),
    }
    true_lat, true_lon, true_depth = 42.80, 13.20, 8.0
    vp, vs = 6.2, 3.3
    origin_sec = 100.0

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        (tmp / 'pk').mkdir()
        with open(tmp / 'station.dat', 'w') as f:
            for sta, (lat, lon) in stations.items():
                f.write(f'{lon:.4f} {lat:.4f} NET {sta} EHZ 0.0\n')
        for sta, (lat, lon) in stations.items():
            p_arrival = origin_sec + _travel_time(true_lat, true_lon, true_depth, lat, lon, vp)
            s_arrival = origin_sec + _travel_time(true_lat, true_lon, true_depth, lat, lon, vs)
            (tmp / 'pk' / f'NET.{sta}.P.txt').write_text(f'{p_arrival:.3f} 1.0 0.0\n')
            (tmp / 'pk' / f'NET.{sta}.S.txt').write_text(f'{s_arrival:.3f} 1.0 0.0\n')

        subprocess.run(
            ['REAL', '-D2016/10/14', '-R0.1/20/0.02/2/5', '-S3/2/5/2/0.5/0.0/1.5',
             f'-V{vp}/{vs}', 'station.dat', 'pk'],
            cwd=tmp, capture_output=True, text=True, timeout=30, check=True,
        )
        catalog = (tmp / 'catalog_sel.txt').read_text()
        assert catalog.strip(), 'REAL produced an empty catalog_sel.txt (event not associated)'
        fields = catalog.split()
        lat, lon, depth = float(fields[7]), float(fields[8]), float(fields[9])

    assert abs(lat - true_lat) < 0.01, f'lat {lat} not within 0.01 deg of true {true_lat}'
    assert abs(lon - true_lon) < 0.01, f'lon {lon} not within 0.01 deg of true {true_lon}'
    assert abs(depth - true_depth) < 1.0, f'depth {depth} not within 1 km of true {true_depth}'
    return f'associated+located at lat={lat:.4f} lon={lon:.4f} depth={depth:.2f} (true 42.80, 13.20, 8.00)'


functional('REAL', _test_real)


# NLLoc: Vel2Grid -> Grid2Time -> NLLoc -----------------------------
# Same synthetic event/station geometry as the REAL test above (not a
# coincidence: lets both tests be checked against each other by hand).
# Builds a 2D travel-time grid wide enough to cover the whole 3D search
# grid plus station offsets (an undersized grid silently rejects
# observations, NLLoc will report "0 events located" with no error at
# default verbosity) and a NLLOC_OBS pick file, then runs the full
# pipeline and reads the recovered hypocenter back out of the .hyp file.
def _test_nlloc():
    stations = {
        'STA1': (42.850, 13.150), 'STA2': (42.760, 13.260),
        'STA3': (42.820, 13.310), 'STA4': (42.740, 13.130),
        'STA5': (42.900, 13.230),
    }
    true_lat, true_lon, true_depth = 42.80, 13.20, 8.0
    vp, vs = 6.2, 3.3
    origin_sec = 100.0

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        for d in ('model', 'time', 'obs', 'loc'):
            (tmp / d).mkdir()

        gtsrce = '\n'.join(
            f'GTSRCE {sta} LATLON {lat:.3f} {lon:.3f} 0 0.0'
            for sta, (lat, lon) in stations.items()
        )
        (tmp / 'test.in').write_text(f'''CONTROL 2 54321
TRANS SIMPLE {true_lat} {true_lon} 0.0

VGOUT ./model/test
VGTYPE P
VGGRID  2 161 45  0.0 0.0 -2.0  0.5 0.5 0.5  SLOW_LEN
LAYER 0.0 {vp} 0.0 {vs} 0.0 2.7 0.0

GTFILES ./model/test ./time/test P
GTMODE GRID2D ANGLES_NO
GT_PLFD 1.0e-3 0

{gtsrce}

LOCSIG geolab-gpu2 functional test
LOCCOM Minimal synthetic single event
LOCFILES ./obs/test.obs NLLOC_OBS ./time/test ./loc/test
LOCHYPOUT SAVE_NLLOC_ALL
LOCSEARCH OCT 10 10 4 0.01 5000 1000 0 1
LOCGRID  81 81 45  -20.0 -20.0 -2.0  0.5 0.5 0.5  PROB_DENSITY SAVE
LOCMETH EDT_OT_WT 9999.0 4 -1 -1 1.68 6 -1.0 1
LOCGAU 0.2 0.0
LOCGAU2 0.02 0.05 2.0
LOCPHASEID P P
LOCQUAL2ERR 0.1 0.5 1.0 2.0 99999.9
''')

        obs_lines = []
        for sta, (lat, lon) in stations.items():
            arrival = origin_sec + _travel_time(true_lat, true_lon, true_depth, lat, lon, vp)
            obs_lines.append(f'{sta} ? ? ? P ? 20260101 0000 {arrival:.3f} GAU 0.05 -1.0 -1.0 -1.0')
        (tmp / 'obs' / 'test.obs').write_text('\n'.join(obs_lines) + '\n')

        for exe in ('Vel2Grid', 'Grid2Time', 'NLLoc'):
            subprocess.run([exe, 'test.in'], cwd=tmp, capture_output=True,
                            text=True, timeout=60, check=True)

        hyp_text = (tmp / 'loc' / 'last.hyp').read_text()
        geo = next(l for l in hyp_text.splitlines() if l.startswith('GEOGRAPHIC')).split()
        ot = geo.index('OT')
        hour, minute, sec = int(geo[ot + 4]), int(geo[ot + 5]), float(geo[ot + 6])
        origin_recovered = hour * 3600 + minute * 60 + sec
        lat = float(geo[geo.index('Lat') + 1])
        lon = float(geo[geo.index('Long') + 1])
        depth = float(geo[geo.index('Depth') + 1])

    assert abs(lat - true_lat) < 0.01, f'lat {lat} not within 0.01 deg of true {true_lat}'
    assert abs(lon - true_lon) < 0.01, f'lon {lon} not within 0.01 deg of true {true_lon}'
    assert abs(depth - true_depth) < 1.0, f'depth {depth} not within 1 km of true {true_depth}'
    assert abs(origin_recovered - origin_sec) < 1.0, (
        f'origin time {origin_recovered}s not within 1s of true {origin_sec}s')
    return (f'located at lat={lat:.4f} lon={lon:.4f} depth={depth:.2f} '
            f'OT={origin_recovered:.2f}s (true 42.80, 13.20, 8.00, 100.00s)')


functional('NLLoc', _test_nlloc)


# hypoDD: catalog-only (IDAT=2) double-difference relocation --------
# Two synthetic events ~140 m apart at the same depth, each started
# from a deliberately wrong initial location (~300-400 m off, in
# event.sel) with no cross-correlation data (dt.cc left empty, IDAT=2
# in hypoDD.inp). dt.ct holds the true P travel time for each event at
# each of 8 stations; 4 stations (and 2 events x 4 unknowns) is
# underdetermined for the double-difference solver, 8 is comfortably
# enough. Passes if hypoDD.reloc ends up close to the true locations,
# not just closer than where it started.
def _test_hypodd():
    events_true = {1: (37.8800, -122.2400, 9.0), 2: (37.8810, -122.2390, 9.0)}
    events_init = {1: (37.8790, -122.2412, 9.0), 2: (37.8822, -122.2378, 9.0)}
    stations = {
        'STA1': (37.9000, -122.2600), 'STA2': (37.8600, -122.2200),
        'STA3': (37.8900, -122.2100), 'STA4': (37.8700, -122.2700),
        'STA5': (37.9100, -122.2200), 'STA6': (37.8500, -122.2600),
        'STA7': (37.8950, -122.2650), 'STA8': (37.8650, -122.2150),
    }
    vp = 6.2

    hypodd_inp = '''hypoDD_2
*--- input file selection
dt.cc
*
dt.ct
*
event.sel
*
station.sel
*--- output file selection
hypoDD.loc
hypoDD.reloc
hypoDD.sta
hypoDD.res
hypoDD.src
*--- data type selection
*   IDAT   IPHA   MAXDIST
     2       1       120
*--- event clustering
* OBSCC  OBSCT   MINDIST  MAXDIST  MAXGAP
      0     0        -999     -999    -999
*--- solution control
*  ISTART  ISOLV  IAQ  NSET
      2       1     1     1
*--- data weighting and re-weighting
* NITER  WTCCP  WTCCS  WRCC  WDCC  WTCTP  WTCTS  WRCT  WDCT  DAMP
    5      0.01   0.005  -999  -999  1.00   0.5   -999  -999   20  ct
*--- 1D model, variable vp/vs ratio
1
0.00 -9
6.20 -9
1.879 -9
*--- event selection
     0
'''

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        with open(tmp / 'station.sel', 'w') as f:
            for sta, (lat, lon) in stations.items():
                f.write(f'{sta} {lat:.4f} {lon:.4f}\n')
        with open(tmp / 'event.sel', 'w') as f:
            for eid, (lat, lon, dep) in events_init.items():
                f.write(f'20260101 12000000 {lat:.4f} {lon:.4f} {dep:.3f} 1.0 0.1 0.1 0.01 {eid}\n')
        with open(tmp / 'dt.ct', 'w') as f:
            f.write('#      1      2\n')
            for sta, (slat, slon) in stations.items():
                t1 = _travel_time(*events_true[1], slat, slon, vp)
                t2 = _travel_time(*events_true[2], slat, slon, vp)
                f.write(f'{sta}  {t1:.4f}  {t2:.4f}  1.0000  P\n')
        (tmp / 'dt.cc').touch()
        (tmp / 'hypoDD.inp').write_text(hypodd_inp)

        subprocess.run(['hypoDD', 'hypoDD.inp'], cwd=tmp, capture_output=True,
                        text=True, timeout=30, check=True)

        recs = {}
        for line in (tmp / 'hypoDD.reloc').read_text().splitlines():
            f_ = line.split()
            recs[int(f_[0])] = (float(f_[1]), float(f_[2]), float(f_[3]))

    def _err_km(a, b):
        dlat = (a[0] - b[0]) * KM_PER_DEG
        dlon = (a[1] - b[1]) * KM_PER_DEG * math.cos(math.radians((a[0] + b[0]) / 2))
        return math.sqrt(dlat ** 2 + dlon ** 2 + (a[2] - b[2]) ** 2)

    assert set(recs) == {1, 2}, f'expected relocations for events 1 and 2, got {sorted(recs)}'
    err1 = _err_km(recs[1], events_true[1])
    err2 = _err_km(recs[2], events_true[2])
    assert err1 < 0.1 and err2 < 0.1, (
        f'relocated events {err1 * 1000:.0f}m / {err2 * 1000:.0f}m from true synthetic '
        f'locations, expected < 100m (initial guesses were ~300-400m off)')
    return f'relocated 2 events to within {max(err1, err2) * 1000:.0f}m of true synthetic locations'


functional('hypoDD', _test_hypodd)


# HypoInverse: reproduce its own bundled reference case exactly -----
# /opt/geolab-tests/hypoinverse (baked in by the Dockerfile) holds
# HypoInverse's own `testone` example files (a real Geysers-area
# earthquake: phases, station list, crustal model) plus run.hyp, a
# lightly-reordered copy of the repo's own testone.hyp command script
# with explicit blank-line answers to its interactive confirmation
# prompts (see the comment above the Dockerfile's `cat > run.hyp` step
# for why the unmodified script silently produces "SKIP BAD PHASE
# CARD" for every pick otherwise) and a trailing LOC/STOP to actually
# run the location. HypoInverse is deterministic (no iterative search
# like REAL/NLLoc use), so re-running it should reproduce the shipped
# testone.sum reference line exactly, not just "close".
def _test_hypoinverse():
    import shutil

    src = Path('/opt/geolab-tests/hypoinverse')
    expected = ('20100103 0833  7.75 38 48.82 122W48.97   2.45 D 2.90 78  '
                '19  1.2 0.06  0.1  0.1 AJ   71329580 3GEY')

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        for name in ('run.hyp', 'testone.phs', 'allseed.sta', 'all2000seed.fmc',
                     'all2000seed.cal', 'gey.crh', 'gey4.del'):
            shutil.copy(src / name, tmp / name)

        with open(tmp / 'run.hyp') as stdin:
            subprocess.run(['hyp1.40'], stdin=stdin, cwd=tmp,
                            capture_output=True, text=True, timeout=30, check=True)

        summary = (tmp / 'testone.sum').read_text().strip()

    assert summary == expected, f'summary {summary!r} != known-correct reference {expected!r}'
    return "reproduced HypoInverse's own testone reference summary line exactly"


functional('hyp1.40', _test_hypoinverse)

## Utilities

In [ ]:
py('tqdm',
   smoke=lambda m: list(m.tqdm(range(3), disable=True)))
py('requests')
py('yaml', alias='pyyaml',
   smoke=lambda m: m.safe_load('a: 1'))
cli('gs', version_flag='--version')      # ghostscript
cli('ffmpeg', version_flag='-version')

## Dev tools

In [ ]:
cli('gh')
cli('gh-scoped-creds')
py('pytest')
cli('ruff')

## Jupyter stack & extensions

In [ ]:
py('jupyterhub')
py('jupyter_server')
py('jupyterlab')
py('ipykernel')
py('jupyter_resource_usage', alias='jupyter-resource-usage')
py('jupyter_ruff', alias='jupyter-ruff')
py('jupyter_server_proxy', alias='jupyter-server-proxy')
py('jupyterlab_git', alias='jupyterlab-git')
py('jupyterlab_myst', alias='jupyterlab-myst')
py('jupyterlab_code_formatter')
py('jupyterlab_pygments')
py('nbdime')

## pip packages & visualization

In [ ]:
# EarthScope --------------------------------------------------
py('earthscope_sdk', alias='earthscope-sdk')
cli('es')                       # earthscope-cli entry point
py('earthscopestraintools')

# Jupyter add-ons ---------------------------------------------
py('jupyterlab_jupyterbook_navigation')
py('jupyterlab_topbar_stop_server', alias='jupyterlab-topbar-stop-server',
   smoke=lambda m: m._jupyter_labextension_paths())

# Visualization & data frames ---------------------------------
py('altair',
   smoke=lambda m: m.Chart())
py('plotly')
py('polars',
   smoke=lambda m: m.DataFrame({'a': [1, 2]}))
py('vegafusion')
py('vl_convert', alias='vl-convert-python')
py('ipympl')
py('hvplot')
py('holoviews', alias='holoviews',
   smoke=lambda m: m.Curve([1, 2, 3]))
py('panel')

## Interactive widgets

In [ ]:
py('ipywidgets',
   smoke=lambda m: m.IntSlider(value=5, min=0, max=10))
py('anywidget')
py('bqplot')
py('ipytree', smoke=lambda m: m.Node(name='root'))
py('ipycytoscape', smoke=lambda m: m.CytoscapeWidget())
py('itables')
py('ipydatagrid')
from sidecar import Sidecar  # noqa: F401
py('sidecar')

## Summary

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(RESULTS,
                  columns=['package', 'status', 'version', 'error'])

passed = int((df['status'] == 'OK').sum())
total = len(df)
failed = total - passed

print(f'Results: {passed}/{total} OK, {failed} failed')
if failed:
    print('\nFailures:')
    for _, row in df[df['status'] == 'FAIL'].iterrows():
        print(f"  {row['package']:35s}  {row['error']}")

df.style.map(
    lambda v: ('color: red; font-weight: bold' if v == 'FAIL'
               else 'color: green'),
    subset=['status']
)